In [58]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_denmark_co2_from_2021_to_april_8():
    base_url = "https://api.energidataservice.dk/dataset/CO2Emis"
    all_records = []
    
    # 1. Exact start date and hardcoded end date
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2026, 4, 8, 23, 59)
    
    print(f"Fetching data from {start_date} to {end_date}...")

    # 2. Fetch in 30-day chunks to prevent API timeout
    current_start = start_date
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=30), end_date)
        
        params = {
            'start': current_start.strftime('%Y-%m-%dT%H:%M'),
            'end': current_end.strftime('%Y-%m-%dT%H:%M'),
            'columns': 'Minutes5DK,PriceArea,CO2Emission',
            'sort': 'Minutes5DK ASC'
        }
        
        response = requests.get(base_url, params=params)
        if response.status_code == 200:
            data = response.json()
            records = data.get('records', [])
            all_records.extend(records)
            print(f"Retrieved {len(records)} records for period starting {current_start.date()}")
        else:
            print(f"Error fetching data for {current_start.date()}: {response.status_code}")
            
        current_start = current_end

    df = pd.DataFrame(all_records)
    
    if not df.empty:
        # Convert to datetime and ensure numeric types
        df['Minutes5DK'] = pd.to_datetime(df['Minutes5DK'])
        df['CO2Emission'] = pd.to_numeric(df['CO2Emission'])
    
    return df

# Execute the extraction
co2_df = fetch_denmark_co2_from_2021_to_april_8()

# Display the first few rows
print("\nFirst 5 rows of data:")
print(co2_df.head())

# Display the last few rows to prove it stops exactly on April 8th!
print("\nLast 5 rows of data:")
print(co2_df.tail())

Fetching data from 2021-01-01 00:00:00 to 2026-04-08 23:59:00...
Retrieved 17280 records for period starting 2021-01-01
Retrieved 17280 records for period starting 2021-01-31
Retrieved 17256 records for period starting 2021-03-02
Retrieved 17280 records for period starting 2021-04-01
Retrieved 17280 records for period starting 2021-05-01
Retrieved 17280 records for period starting 2021-05-31
Retrieved 17280 records for period starting 2021-06-30
Retrieved 17280 records for period starting 2021-07-30
Retrieved 17280 records for period starting 2021-08-29
Retrieved 17280 records for period starting 2021-09-28
Retrieved 17280 records for period starting 2021-10-28
Retrieved 17280 records for period starting 2021-11-27
Retrieved 17280 records for period starting 2021-12-27
Retrieved 17280 records for period starting 2022-01-26
Retrieved 17280 records for period starting 2022-02-25
Retrieved 17256 records for period starting 2022-03-27
Retrieved 17280 records for period starting 2022-04-26


In [59]:
co2_df

,Minutes5DK,PriceArea,CO2Emission
0,2021-01-01 00:00:00,DK2,218.0
1,2021-01-01 00:00:00,DK1,218.0
2,2021-01-01 00:05:00,DK2,199.0
3,2021-01-01 00:05:00,DK1,199.0
4,2021-01-01 00:10:00,DK2,188.0
...,...,...,...
1106907,2026-04-08 23:40:00,DK2,40.0
1106908,2026-04-08 23:45:00,DK1,147.0
1106909,2026-04-08 23:45:00,DK2,40.0
1106910,2026-04-08 23:50:00,DK1,148.0


In [60]:
# 1. Make sure pandas knows the column is a datetime object
co2_df['Minutes5DK'] = pd.to_datetime(co2_df['Minutes5DK'])

# 2. Group by DK1/DK2, resample to 1 hour ('1h'), and get the mean
hourly_co2 = co2_df.groupby('PriceArea').resample('1h', on='Minutes5DK').mean().reset_index()

# 3. Rename columns so it is easy to read
hourly_co2.columns = ['PriceArea', 'Time', 'Hourly_CO2_Average']

print(hourly_co2.head())

  PriceArea                Time  Hourly_CO2_Average
0       DK1 2021-01-01 00:00:00          190.833333
1       DK1 2021-01-01 01:00:00          196.000000
2       DK1 2021-01-01 02:00:00          168.166667
3       DK1 2021-01-01 03:00:00          160.083333
4       DK1 2021-01-01 04:00:00          161.083333


In [61]:
hourly_co2['PriceArea'].value_counts()

PriceArea
DK1    46176
DK2    46176
Name: count, dtype: int64

In [67]:
hourly_co2

,PriceArea,Time,Hourly_CO2_Average
0,DK1,2021-01-01 00:00:00,190.833333
1,DK1,2021-01-01 01:00:00,196.000000
2,DK1,2021-01-01 02:00:00,168.166667
3,DK1,2021-01-01 03:00:00,160.083333
4,DK1,2021-01-01 04:00:00,161.083333
...,...,...,...
92347,DK2,2026-04-08 19:00:00,49.333333
92348,DK2,2026-04-08 20:00:00,67.083333
92349,DK2,2026-04-08 21:00:00,74.500000
92350,DK2,2026-04-08 22:00:00,48.833333


In [69]:
hourly_co2[hourly_co2['Hourly_CO2_Average'].isna()]   

,PriceArea,Time,Hourly_CO2_Average
2066,DK1,2021-03-28 02:00:00,NaN
10802,DK1,2022-03-27 02:00:00,NaN
19538,DK1,2023-03-26 02:00:00,NaN
28442,DK1,2024-03-31 02:00:00,NaN
37178,DK1,2025-03-30 02:00:00,NaN
...,...,...,...
89060,DK2,2025-11-22 20:00:00,NaN
89061,DK2,2025-11-22 21:00:00,NaN
89062,DK2,2025-11-22 22:00:00,NaN
89063,DK2,2025-11-22 23:00:00,NaN


In [62]:
import requests
import pandas as pd
from datetime import datetime

def fetch_weather_from_2021_to_april_8():
    start_date = "2021-01-01"
    end_date = "2026-04-08"  # Hardcoded end date!
    
    # Rough coordinates for Jutland (DK1) and Zealand (DK2)
    coords = {
        'DK1': {'lat': 56.15, 'lon': 9.50},
        'DK2': {'lat': 55.67, 'lon': 12.00}
    }
    
    all_weather = []
    print(f"🌤️ Fetching Weather from {start_date} to {end_date}...")
    
    for area, loc in coords.items():
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": loc['lat'],
            "longitude": loc['lon'],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "wind_speed_10m,shortwave_radiation"
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame({
                'Time': pd.to_datetime(data['hourly']['time']),
                'PriceArea': area,
                'wind_speed': data['hourly']['wind_speed_10m'],
                'solar_radiation': data['hourly']['shortwave_radiation']
            })
            all_weather.append(df)
            print(f"✅ Retrieved {len(df)} weather records for {area}")
        else:
            print(f"❌ Error for {area}: {response.status_code}")
            
    # Combine DK1 and DK2 into one dataframe
    if all_weather:
        weather_df = pd.concat(all_weather, ignore_index=True)
        return weather_df
    return pd.DataFrame()

# Execute the extraction
weather_df = fetch_weather_from_2021_to_april_8()

# Display the first and last few rows
print("\nFirst 5 rows:")
print(weather_df.head())
print("\nLast 5 rows:")
print(weather_df.tail())

🌤️ Fetching Weather from 2021-01-01 to 2026-04-08...
✅ Retrieved 46176 weather records for DK1
✅ Retrieved 46176 weather records for DK2

First 5 rows:
                 Time PriceArea  wind_speed  solar_radiation
0 2021-01-01 00:00:00       DK1         5.2              0.0
1 2021-01-01 01:00:00       DK1         4.8              0.0
2 2021-01-01 02:00:00       DK1         6.5              0.0
3 2021-01-01 03:00:00       DK1         6.6              0.0
4 2021-01-01 04:00:00       DK1         5.6              0.0

Last 5 rows:
                     Time PriceArea  wind_speed  solar_radiation
92347 2026-04-08 19:00:00       DK2         3.9              0.0
92348 2026-04-08 20:00:00       DK2         3.1              0.0
92349 2026-04-08 21:00:00       DK2         6.8              0.0
92350 2026-04-08 22:00:00       DK2         8.9              0.0
92351 2026-04-08 23:00:00       DK2         9.0              0.0


In [63]:
weather_df

,Time,PriceArea,wind_speed,solar_radiation
0,2021-01-01 00:00:00,DK1,5.2,0.0
1,2021-01-01 01:00:00,DK1,4.8,0.0
2,2021-01-01 02:00:00,DK1,6.5,0.0
3,2021-01-01 03:00:00,DK1,6.6,0.0
4,2021-01-01 04:00:00,DK1,5.6,0.0
...,...,...,...,...
92347,2026-04-08 19:00:00,DK2,3.9,0.0
92348,2026-04-08 20:00:00,DK2,3.1,0.0
92349,2026-04-08 21:00:00,DK2,6.8,0.0
92350,2026-04-08 22:00:00,DK2,8.9,0.0


In [64]:
# 1. Force both to be pandas datetime objects
hourly_co2['Time'] = pd.to_datetime(hourly_co2['Time'])
weather_df['Time'] = pd.to_datetime(weather_df['Time'])

# 2. Strip away any timezone information (tz_localize(None) makes them "naive" datetimes)
# Try/except is used just in case they are already timezone-naive
try:
    hourly_co2['Time'] = hourly_co2['Time'].dt.tz_localize(None)
except TypeError:
    pass # Already timezone-naive

try:
    weather_df['Time'] = weather_df['Time'].dt.tz_localize(None)
except TypeError:
    pass # Already timezone-naive

# 3. Prove it
print("CO2 Time Format:    ", hourly_co2['Time'].dtype)
print("Weather Time Format:", weather_df['Time'].dtype)

CO2 Time Format:     datetime64[ns]
Weather Time Format: datetime64[ns]


In [65]:
# Merge the two dataframes on BOTH Time and PriceArea
master_df = pd.merge(hourly_co2, weather_df, on=['Time', 'PriceArea'], how='inner')

# Sort it neatly so it flows perfectly from 2021 to 2026
master_df = master_df.sort_values(by=['Time', 'PriceArea']).reset_index(drop=True)

print("✅ Data Merged! Here are the first 5 rows of your master dataset:")
print(master_df.head())

✅ Data Merged! Here are the first 5 rows of your master dataset:
  PriceArea                Time  Hourly_CO2_Average  wind_speed  \
0       DK1 2021-01-01 00:00:00          190.833333         5.2   
1       DK2 2021-01-01 00:00:00          190.833333         9.0   
2       DK1 2021-01-01 01:00:00          196.000000         4.8   
3       DK2 2021-01-01 01:00:00          196.000000         8.5   
4       DK1 2021-01-01 02:00:00          168.166667         6.5   

   solar_radiation  
0              0.0  
1              0.0  
2              0.0  
3              0.0  
4              0.0  


In [70]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92352 entries, 0 to 92351
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   PriceArea           92352 non-null  object        
 1   Time                92352 non-null  datetime64[ns]
 2   Hourly_CO2_Average  92254 non-null  float64       
 3   wind_speed          92352 non-null  float64       
 4   solar_radiation     92352 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 3.5+ MB


In [71]:
master_df.to_csv('data/master_dataset.csv', index=False)

In [72]:
master_df

,PriceArea,Time,Hourly_CO2_Average,wind_speed,solar_radiation
0,DK1,2021-01-01 00:00:00,190.833333,5.2,0.0
1,DK2,2021-01-01 00:00:00,190.833333,9.0,0.0
2,DK1,2021-01-01 01:00:00,196.000000,4.8,0.0
3,DK2,2021-01-01 01:00:00,196.000000,8.5,0.0
4,DK1,2021-01-01 02:00:00,168.166667,6.5,0.0
...,...,...,...,...,...
92347,DK2,2026-04-08 21:00:00,74.500000,6.8,0.0
92348,DK1,2026-04-08 22:00:00,129.583333,10.4,0.0
92349,DK2,2026-04-08 22:00:00,48.833333,8.9,0.0
92350,DK1,2026-04-08 23:00:00,148.909091,10.7,0.0


In [73]:
import requests
import pandas as pd
import time
import calendar
from datetime import datetime

def fetch_all_historical_prices_to_april_8():
    all_prices = []
    years = [2021, 2022, 2023, 2024, 2025, 2026]
    print("💰 Fetching 5 Years of Prices up to April 8, 2026...")
    
    for year in years:
        # Stop at April for 2026
        max_month = 4 if year == 2026 else 12
        
        for month in range(1, max_month + 1):
            # Hardcode the cutoff day for April 2026
            if year == 2026 and month == 4:
                last_day = 8
            else:
                last_day = calendar.monthrange(year, month)[1]
                
            start = f"{year}-{month:02d}-01T00:00"
            end = f"{year}-{month:02d}-{last_day:02d}T23:59"
            
            params = {"start": start, "end": end, "filter": '{"PriceArea": ["DK1", "DK2"]}', "limit": 100000}
            
            # 1. Try Old API (Elspotprices)
            try:
                r1 = requests.get("https://api.energidataservice.dk/dataset/Elspotprices", params=params)
                if r1.status_code == 200:
                    for row in r1.json().get('records', []):
                        all_prices.append({'Time': row['HourUTC'], 'PriceArea': row['PriceArea'], 'spot_price_dkk_mwh': row['SpotPriceDKK']})
            except Exception: pass
            
            # 2. Try New API (DayAheadPrices)
            try:
                r2 = requests.get("https://api.energidataservice.dk/dataset/DayAheadPrices", params=params)
                if r2.status_code == 200:
                    for row in r2.json().get('records', []):
                        all_prices.append({'Time': row['TimeUTC'], 'PriceArea': row['PriceArea'], 'spot_price_dkk_mwh': row['DayAheadPriceDKK']})
            except Exception: pass
                    
            time.sleep(0.5) # Be polite to the API

    df = pd.DataFrame(all_prices)
    df['Time'] = pd.to_datetime(df['Time']).dt.tz_localize(None)
    
    # Drop overlapping duplicates
    df = df.drop_duplicates(subset=['Time', 'PriceArea'])
    return df

# 1. Fetch the Prices
price_df = fetch_all_historical_prices_to_april_8()

# 2. Final Master Merge!
final_df = pd.merge(master_df, price_df, on=['Time', 'PriceArea'], how='inner')

# 3. Rename columns to exactly match your Postgres Database schema
final_df = final_df.rename(columns={
    'Time': 'ds',
    'PriceArea': 'price_area',
    'Hourly_CO2_Average': 'co2_emissions_g_kwh'
})

# Reorder columns so they look clean
final_df = final_df[['ds', 'price_area', 'spot_price_dkk_mwh', 'co2_emissions_g_kwh', 'wind_speed', 'solar_radiation']]

print("\n🏆 FINAL DATASET COMPLETE! Here are the first 5 rows:")
print(final_df.head())

print("\n🛑 Checking the last 5 rows to ensure it stops exactly on April 8th:")
print(final_df.tail())

💰 Fetching 5 Years of Prices up to April 8, 2026...

🏆 FINAL DATASET COMPLETE! Here are the first 5 rows:
                   ds price_area  spot_price_dkk_mwh  co2_emissions_g_kwh  \
0 2021-01-01 00:00:00        DK1          358.579987           190.833333   
1 2021-01-01 00:00:00        DK2          358.579987           190.833333   
2 2021-01-01 01:00:00        DK1          332.459991           196.000000   
3 2021-01-01 01:00:00        DK2          332.459991           196.000000   
4 2021-01-01 02:00:00        DK1          319.369995           168.166667   

   wind_speed  solar_radiation  
0         5.2              0.0  
1         9.0              0.0  
2         4.8              0.0  
3         8.5              0.0  
4         6.5              0.0  

🛑 Checking the last 5 rows to ensure it stops exactly on April 8th:
                       ds price_area  spot_price_dkk_mwh  co2_emissions_g_kwh  \
92343 2026-04-08 19:00:00        DK2         1157.639685            49.333333   
92

In [74]:
final_df

,ds,price_area,spot_price_dkk_mwh,co2_emissions_g_kwh,wind_speed,solar_radiation
0,2021-01-01 00:00:00,DK1,358.579987,190.833333,5.2,0.0
1,2021-01-01 00:00:00,DK2,358.579987,190.833333,9.0,0.0
2,2021-01-01 01:00:00,DK1,332.459991,196.000000,4.8,0.0
3,2021-01-01 01:00:00,DK2,332.459991,196.000000,8.5,0.0
4,2021-01-01 02:00:00,DK1,319.369995,168.166667,6.5,0.0
...,...,...,...,...,...,...
92343,2026-04-08 19:00:00,DK2,1157.639685,49.333333,3.9,0.0
92344,2026-04-08 20:00:00,DK1,951.249272,107.000000,8.9,0.0
92345,2026-04-08 20:00:00,DK2,951.249272,67.083333,3.1,0.0
92346,2026-04-08 21:00:00,DK1,877.047347,100.750000,8.7,0.0


In [75]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92348 entries, 0 to 92347
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ds                   92348 non-null  datetime64[ns]
 1   price_area           92348 non-null  object        
 2   spot_price_dkk_mwh   92348 non-null  float64       
 3   co2_emissions_g_kwh  92250 non-null  float64       
 4   wind_speed           92348 non-null  float64       
 5   solar_radiation      92348 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 4.2+ MB


In [ ]:
'''
We will tell the computer to use both methods automatically based on how broken the API is.

Rule 1: If the gap is small (1 to 3 hours, like Daylight Saving Time), draw a clean straight line.

Rule 2: If the gap is massive (API crashed for a whole day), go back exactly one week and copy the seasonal data.
'''
print("🔧 Running the Hybrid Data Imputation Patch...")

# Step 1: Patch the small holes (Interpolate up to 3 hours max)
final_df['co2_emissions_g_kwh'] = final_df.groupby('price_area')['co2_emissions_g_kwh'].transform(
    lambda x: x.interpolate(method='linear', limit=3)
)

# Step 2: Patch the massive craters (Copy data from exactly 168 hours / 1 week ago)
final_df['co2_emissions_g_kwh'] = final_df.groupby('price_area')['co2_emissions_g_kwh'].transform(
    lambda x: x.fillna(x.shift(168))
)

# Step 3: Verify the AI physics engine is flawless
missing_co2 = final_df['co2_emissions_g_kwh'].isna().sum()

if missing_co2 == 0:
    print("✅ SUCCESS: Zero missing values! The timeline is perfectly intact.")
else:
    print(f"⚠️ WARNING: Still have {missing_co2} missing rows.")

# Prove it by printing the info again!
final_df.info()

🔧 Running the Hybrid Data Imputation Patch...
✅ SUCCESS: Zero missing values! The timeline is perfectly intact.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92348 entries, 0 to 92347
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ds                   92348 non-null  datetime64[ns]
 1   price_area           92348 non-null  object        
 2   spot_price_dkk_mwh   92348 non-null  float64       
 3   co2_emissions_g_kwh  92348 non-null  float64       
 4   wind_speed           92348 non-null  float64       
 5   solar_radiation      92348 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 4.2+ MB


In [80]:
# Convert MWh to kWh by dividing by 1000
final_df['spot_price_dkk_mwh'] = final_df['spot_price_dkk_mwh'] / 1000

# Rename the column so we don't confuse ourselves later
final_df = final_df.rename(columns={'spot_price_dkk_mwh': 'spot_price_dkk_kwh'})

print("✅ Price converted to DKK/kWh! Here is the updated look:")
print(final_df[['ds', 'spot_price_dkk_kwh', 'co2_emissions_g_kwh']].head())

✅ Price converted to DKK/kWh! Here is the updated look:
                   ds  spot_price_dkk_kwh  co2_emissions_g_kwh
0 2021-01-01 00:00:00             0.35858           190.833333
1 2021-01-01 00:00:00             0.35858           190.833333
2 2021-01-01 01:00:00             0.33246           196.000000
3 2021-01-01 01:00:00             0.33246           196.000000
4 2021-01-01 02:00:00             0.31937           168.166667


In [81]:
final_df.to_csv('data/final_master_dataset.csv', index=False)

In [82]:
final_df

,ds,price_area,spot_price_dkk_kwh,co2_emissions_g_kwh,wind_speed,solar_radiation
0,2021-01-01 00:00:00,DK1,0.358580,190.833333,5.2,0.0
1,2021-01-01 00:00:00,DK2,0.358580,190.833333,9.0,0.0
2,2021-01-01 01:00:00,DK1,0.332460,196.000000,4.8,0.0
3,2021-01-01 01:00:00,DK2,0.332460,196.000000,8.5,0.0
4,2021-01-01 02:00:00,DK1,0.319370,168.166667,6.5,0.0
...,...,...,...,...,...,...
92343,2026-04-08 19:00:00,DK2,1.157640,49.333333,3.9,0.0
92344,2026-04-08 20:00:00,DK1,0.951249,107.000000,8.9,0.0
92345,2026-04-08 20:00:00,DK2,0.951249,67.083333,3.1,0.0
92346,2026-04-08 21:00:00,DK1,0.877047,100.750000,8.7,0.0


In [83]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92348 entries, 0 to 92347
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ds                   92348 non-null  datetime64[ns]
 1   price_area           92348 non-null  object        
 2   spot_price_dkk_kwh   92348 non-null  float64       
 3   co2_emissions_g_kwh  92348 non-null  float64       
 4   wind_speed           92348 non-null  float64       
 5   solar_radiation      92348 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 4.2+ MB


In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/praful/Documents/Aalborg_University/2nd_Sem/Data_Engineering_and_MLOps/final/greenhour/data/final_master_dataset.csv')

In [4]:
df.head()

,ds,price_area,spot_price_dkk_kwh,co2_emissions_g_kwh,wind_speed,solar_radiation
0,2021-01-01 00:00:00,DK1,0.35858,190.833333,5.2,0.0
1,2021-01-01 00:00:00,DK2,0.35858,190.833333,9.0,0.0
2,2021-01-01 01:00:00,DK1,0.33246,196.000000,4.8,0.0
3,2021-01-01 01:00:00,DK2,0.33246,196.000000,8.5,0.0
4,2021-01-01 02:00:00,DK1,0.31937,168.166667,6.5,0.0


In [7]:
print("\nDK1 Statistics:")
df[df['price_area'] == 'DK1'].describe()



DK1 Statistics:


,spot_price_dkk_kwh,co2_emissions_g_kwh,wind_speed,solar_radiation
count,46174.000000,46174.000000,46174.000000,46174.000000
mean,0.809074,115.790903,14.580550,116.818166
std,0.715094,69.152695,6.738626,187.878794
min,-3.277390,-213.083333,0.000000,0.000000
25%,0.414055,61.250000,9.400000,0.000000
50%,0.665795,104.500000,13.800000,4.000000
75%,0.945803,158.583333,18.800000,168.000000
max,6.982420,464.750000,52.000000,863.000000


In [8]:
print("\nDK2 Statistics:")
df[df['price_area'] == 'DK2'].describe()


DK2 Statistics:


,spot_price_dkk_kwh,co2_emissions_g_kwh,wind_speed,solar_radiation
count,46174.000000,46174.000000,46174.000000,46174.000000
mean,0.792948,97.294700,15.438002,125.268766
std,0.720644,65.836550,7.549240,198.860182
min,-0.447460,8.083333,0.000000,0.000000
25%,0.369904,49.000000,9.700000,0.000000
50%,0.648770,76.500000,14.500000,4.000000
75%,0.948807,130.833333,20.100000,182.000000
max,6.982640,581.333333,54.600000,862.000000
